In [0]:
%run /Workspace/Users/shreyash270204@outlook.com/databricks/utilities/config.py

Config loaded. STORAGE_ACCOUNT=financestorage1 SNAPSHOT_DATE=latest per exchange TAXONOMY_VERSION=v1


In [0]:
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable
import uuid

PIPELINE_RUN_ID = str(uuid.uuid4())
print(f"pipeline_run_id: {PIPELINE_RUN_ID}")

pipeline_run_id: 2ccac2c2-6882-4f26-8fc9-f621d06509b8


In [0]:
financial_instrument = spark.read.format("delta").load(silver_path("financial_instrument"))
dim_exchange = spark.read.format("delta").load(silver_path("dim_exchange"))
dim_country = spark.read.format("delta").load(silver_path("dim_country"))
dim_currency = spark.read.format("delta").load(silver_path("dim_currency"))
taxonomy_mapping = spark.read.format("delta").load(silver_path("taxonomy_mapping")).where("is_active = true")

equity_ext = spark.read.format("delta").load(silver_path("equity_extension"))
etf_ext = spark.read.format("delta").load(silver_path("etf_extension"))
fund_ext = spark.read.format("delta").load(silver_path("fund_extension"))

total_records = financial_instrument.count()
print(f"financial_instrument total: {total_records}")

financial_instrument total: 66105


In [0]:
completeness_cols = ["symbol", "instrument_name", "asset_type", "country", "currency"]
null_counts = financial_instrument.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in completeness_cols]
).first().asDict()

print("=== Completeness (null %) ===")
for c in completeness_cols:
    pct = round(100 * null_counts[c] / total_records, 2)
    print(f"  {c}: {pct}% null")

=== Completeness (null %) ===
  symbol: 0.0% null
  instrument_name: 0.57% null
  asset_type: 0.0% null
  country: 3.29% null
  currency: 0.61% null


In [0]:
valid_exchanges = [r["exchange_code"] for r in dim_exchange.select("exchange_code").collect()]
valid_countries = [r["country_name"] for r in dim_country.select("country_name").collect()]
valid_currencies = [r["currency_code"] for r in dim_currency.select("currency_code").collect()]

record_json = F.to_json(F.struct(
    "instrument_id", "symbol", "instrument_name", "asset_type",
    "country", "currency", "exchange", "source_dataset", "snapshot_date"
))

def failing(condition, reason, column):
    return (
        financial_instrument.where(condition)
        .select(
            F.col("source_dataset"),
            record_json.alias("source_record"),
            F.lit(reason).alias("failure_reason"),
            F.lit(column).alias("failed_column"),
            F.lit(PIPELINE_RUN_ID).alias("pipeline_run_id"),
            F.col("snapshot_date"),
        )
    )

invalid_parts = [
    failing(F.col("symbol").isNull(), "CORE_FIELD_MISSING", "symbol"),
    failing(F.col("instrument_name").isNull(), "CORE_FIELD_MISSING", "instrument_name"),
    failing(F.col("asset_type").isNull(), "CORE_FIELD_MISSING", "asset_type"),
    failing(~F.col("exchange").isin(valid_exchanges), "INVALID_EXCHANGE", "exchange"),
    failing(F.col("country").isNotNull() & ~F.col("country").isin(valid_countries), "INVALID_COUNTRY", "country"),
    failing(F.col("currency").isNotNull() & ~F.col("currency").isin(valid_currencies), "INVALID_CURRENCY", "currency"),
]

invalid_from_validity = invalid_parts[0]
for p in invalid_parts[1:]:
    invalid_from_validity = invalid_from_validity.unionByName(p)

print(f"Validity failure rows: {invalid_from_validity.count()}")

Validity failure rows: 379


In [0]:
def find_duplicates(asset_class: str):
    frames = []
    for exch, classes in EXCHANGE_ASSET_COVERAGE.items():
        if asset_class not in classes:
            continue
        path = latest_snapshot_path(asset_class, exch)
        if path is None:
            continue
        df = (
            spark.read.option("header", True).option("inferSchema", True)
            .option("multiLine", True).option("escape", "\"")
            .csv(f"{path}/*.csv")
            .withColumn("exchange", F.coalesce(F.col("exchange"), F.lit(exch)))
            .withColumn("_snapshot_date", F.lit(path.split("snapshot_date=")[-1]))
        )
        frames.append(df)
    if not frames:
        return None
    out = frames[0]
    for f in frames[1:]:
        out = out.unionByName(f, allowMissingColumns=True)

    w = Window.partitionBy("symbol", "exchange").orderBy(F.lit(1))
    ranked = out.withColumn("_rn", F.row_number().over(w))
    dups = ranked.filter("_rn > 1").drop("_rn")

    return dups.select(
        F.lit(asset_class).alias("source_dataset"),
        F.to_json(F.struct("symbol", "exchange", "name", "currency", "_snapshot_date")).alias("source_record"),
        F.lit("DUPLICATE_RECORD").alias("failure_reason"),
        F.lit("symbol+exchange").alias("failed_column"),
        F.lit(PIPELINE_RUN_ID).alias("pipeline_run_id"),
        F.col("_snapshot_date").alias("snapshot_date"),
    )

dup_frames = []
duplicate_counts = {}
for ac in ASSET_CLASSES:
    d = find_duplicates(ac)
    c = d.count() if d is not None else 0
    duplicate_counts[ac] = c
    if d is not None and c > 0:
        dup_frames.append(d)

print(f"Duplicate counts by dataset: {duplicate_counts}")

Duplicate counts by dataset: {'equities': 0, 'etfs': 0, 'funds': 0}


In [0]:
all_invalid = invalid_from_validity
for d in dup_frames:
    all_invalid = all_invalid.unionByName(d)

all_invalid = all_invalid.withColumn("quarantine_timestamp", F.current_timestamp())

target_path = quarantine_path("invalid_records")
if DeltaTable.isDeltaTable(spark, target_path):
    target = DeltaTable.forPath(spark, target_path)
    (
        target.alias("t")
        .merge(
            all_invalid.alias("s"),
            "t.source_dataset = s.source_dataset AND t.source_record = s.source_record AND t.failure_reason = s.failure_reason"
        )
        .whenMatchedUpdate(set={"pipeline_run_id": "s.pipeline_run_id", "quarantine_timestamp": "s.quarantine_timestamp"})
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    all_invalid.write.format("delta").mode("overwrite").save(target_path)

invalid_records_df = spark.read.format("delta").load(target_path)
print(f"quarantine/invalid_records: {invalid_records_df.count()} total rows")

quarantine/invalid_records: 379 total rows


In [0]:
VALID_THEMES = ["Technology", "Healthcare", "Energy", "Financial Services", "Industrials", "Consumer", "Real Estate", "Utilities", "Materials"]

def classify_taxonomy(ext_df, asset_type, source_field):
    mapping = taxonomy_mapping.where((F.col("asset_type") == asset_type) & (F.col("source_field") == source_field))
    joined = (
        ext_df.select("instrument_id", F.col(source_field).alias("source_value"))
        .join(mapping.select("source_value", "normalized_theme", "confidence_score"), on="source_value", how="left")
    )
    return (
        joined
        .withColumn(
            "quality_flag",
            F.when(F.col("source_value").isNull(), F.lit(None))
             .when(F.col("normalized_theme").isNull() & F.col("confidence_score").isNull(), F.lit("UNMAPPED_CLASSIFICATION"))
             .when(F.col("normalized_theme").isNull() & F.col("confidence_score").isNotNull(), F.lit("UNKNOWN_CATEGORY"))
             .when(~F.col("normalized_theme").isin(VALID_THEMES), F.lit("INVALID_THEME"))
             .when(F.col("confidence_score") < 0.75, F.lit("LOW_CONFIDENCE_MAPPING"))
             .otherwise(F.lit("OK"))
        )
        .withColumn("asset_type", F.lit(asset_type))
        .withColumn("source_field", F.lit(source_field))
    )

taxonomy_checks = (
    classify_taxonomy(equity_ext, "equity", "sector")
    .unionByName(classify_taxonomy(equity_ext, "equity", "industry_group"))
    .unionByName(classify_taxonomy(etf_ext, "etf", "category_group"))
    .unionByName(classify_taxonomy(etf_ext, "etf", "category"))
    .unionByName(classify_taxonomy(fund_ext, "fund", "category_group"))
    .unionByName(classify_taxonomy(fund_ext, "fund", "category"))
)

print("=== Taxonomy quality flag counts ===")
taxonomy_checks.groupBy("quality_flag").count().show()

=== Taxonomy quality flag counts ===
+--------------------+-----+
|        quality_flag|count|
+--------------------+-----+
|                NULL|17852|
|                  OK|81192|
|LOW_CONFIDENCE_MA...|  659|
|    UNKNOWN_CATEGORY|32507|
+--------------------+-----+



In [0]:
quarantine_taxonomy = (
    taxonomy_checks.where(F.col("quality_flag").isin("UNMAPPED_CLASSIFICATION", "INVALID_THEME"))
    .withColumn("source_dataset", F.col("asset_type"))
    .withColumn("source_record", F.to_json(F.struct("instrument_id", "asset_type", "source_field", "source_value")))
    .withColumn("failure_reason", F.col("quality_flag"))
    .withColumn("failed_column", F.col("source_field"))
    .withColumn("pipeline_run_id", F.lit(PIPELINE_RUN_ID))
    .withColumn("snapshot_date", F.lit(None).cast("string"))
    .withColumn("quarantine_timestamp", F.current_timestamp())
    .select("source_dataset", "source_record", "failure_reason", "failed_column", "pipeline_run_id", "snapshot_date", "quarantine_timestamp")
)

target_path2 = quarantine_path("unmapped_classifications")
if DeltaTable.isDeltaTable(spark, target_path2):
    target2 = DeltaTable.forPath(spark, target_path2)
    (
        target2.alias("t")
        .merge(
            quarantine_taxonomy.alias("s"),
            "t.source_dataset = s.source_dataset AND t.source_record = s.source_record AND t.failure_reason = s.failure_reason"
        )
        .whenMatchedUpdate(set={"pipeline_run_id": "s.pipeline_run_id", "quarantine_timestamp": "s.quarantine_timestamp"})
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    quarantine_taxonomy.write.format("delta").mode("overwrite").save(target_path2)

unmapped_df = spark.read.format("delta").load(target_path2)
print(f"quarantine/unmapped_classifications: {unmapped_df.count()} total rows")

quarantine/unmapped_classifications: 0 total rows


In [0]:
def dataset_metrics(asset_class, asset_type):
    total = financial_instrument.where(F.col("asset_type") == asset_type).count()
    invalid = invalid_records_df.where(F.col("source_dataset") == asset_class).select("source_record").distinct().count()
    dup = duplicate_counts.get(asset_class, 0)
    unmapped = (
        taxonomy_checks.where((F.col("asset_type") == asset_type) & (F.col("quality_flag").isin("UNMAPPED_CLASSIFICATION", "INVALID_THEME")))
        .select("instrument_id").distinct().count()
    )
    low_conf = (
        taxonomy_checks.where((F.col("asset_type") == asset_type) & (F.col("quality_flag") == "LOW_CONFIDENCE_MAPPING"))
        .select("instrument_id").distinct().count()
    )
    valid = total - invalid
    quality_score = round(100 * valid / total, 2) if total > 0 else 0.0
    return {
        "run_id": PIPELINE_RUN_ID,
        "dataset": asset_class,
        "total_records": total,
        "valid_records": valid,
        "invalid_records": invalid,
        "unmapped_records": unmapped,
        "low_confidence_records": low_conf,
        "duplicate_records": dup,
        "quality_score": quality_score,
        "pipeline_status": "SUCCESS",
    }

dq_rows = [dataset_metrics(ac, at) for ac, at in [("equities", "equity"), ("etfs", "etf"), ("funds", "fund")]]
dq_df = spark.createDataFrame(dq_rows).withColumn("run_timestamp", F.current_timestamp())

dq_df.write.format("delta").mode("append").save(gold_path("data_quality"))

print("=== gold_data_quality (this run) ===")
display(dq_df)

=== gold_data_quality (this run) ===


dataset,duplicate_records,invalid_records,low_confidence_records,pipeline_status,quality_score,run_id,total_records,unmapped_records,valid_records,run_timestamp
equities,0,375,657,SUCCESS,99.08,2ccac2c2-6882-4f26-8fc9-f621d06509b8,40549,0,40174,2026-08-21T11:10:07.189Z
etfs,0,0,0,SUCCESS,100.0,2ccac2c2-6882-4f26-8fc9-f621d06509b8,13782,0,13782,2026-08-21T11:10:07.189Z
funds,0,4,2,SUCCESS,99.97,2ccac2c2-6882-4f26-8fc9-f621d06509b8,11774,0,11770,2026-08-21T11:10:07.189Z


In [0]:
invalid_records_df.where(F.col("source_dataset") == "equities").groupBy("failure_reason").count().show()

+------------------+-----+
|    failure_reason|count|
+------------------+-----+
|CORE_FIELD_MISSING|  375|
+------------------+-----+

